# Stable Diffusion Homework (Food-101)

This notebook is a copy of `dl_homework.ipynb` configured for **Food-101** (RGB, 101 classes) via `torchvision.datasets.ImageFolder`.

**Dataset:** download [Food-101 on Kaggle](https://www.kaggle.com/datasets/dansbecker/food-101), unzip, and point `FOOD101_ROOT` in the next cell at the folder that contains the `images/` directory (class subfolders live under `images/`).

Homework alignment:

- RGB color images, **class-conditioned** latent diffusion (same architecture style as `StableDiffusion2026.ipynb`)
- Training from scratch; checkpoints are separate from the CIFAR-10 notebook
- PyTorch + torchvision only (no sklearn / HF / etc.)


In [ ]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Food-101 at 64x64 (adjust if you have more VRAM)
IMAGE_SIZE = 64
LATENT_CHANNELS = 4
LATENT_SIZE = IMAGE_SIZE // 4
BATCH_SIZE = 32
AUTOENCODER_EPOCHS = 15
DIFFUSION_EPOCHS = 40
LR_AUTOENCODER = 2e-4
LR_DIFFUSION = 2e-4
KL_WEIGHT = 1e-4
SIGMA = 25.0
NUM_DIFFUSION_STEPS = 300
MAX_TRAIN_SAMPLES = None  # e.g. 10_000 for faster dry runs
NUM_WORKERS = 2

DATA_ROOT = Path("./data")
# Folder that contains `images/` with one subdirectory per class (standard Food-101 layout)
FOOD101_ROOT = DATA_ROOT / "food-101"

CHECKPOINT_DIR = Path("./checkpoints/stable_diffusion_hw_food101")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

AUTOENCODER_CKPT = CHECKPOINT_DIR / "food101_vae.pt"
DIFFUSION_CKPT = CHECKPOINT_DIR / "food101_latent_diffusion.pt"

# Filled after ImageFolder loads (cell 2)
CLASS_NAMES: list[str] = []
NUM_CLASSES = 0


def denorm(x: torch.Tensor) -> torch.Tensor:
    return x.clamp(-1, 1).add(1).div(2)


def show_tensor_images(images: torch.Tensor, nrow: int = 8, title: str | None = None) -> None:
    grid = torchvision.utils.make_grid(denorm(images.detach().cpu()), nrow=nrow)
    plt.figure(figsize=(12, 6))
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


In [ ]:
FOOD101_IMAGES = FOOD101_ROOT / "images"
if not FOOD101_IMAGES.is_dir():
    raise FileNotFoundError(
        f"Food-101 images folder not found: {FOOD101_IMAGES}. "
        "Download from Kaggle (dansbecker/food-101), unzip, and set FOOD101_ROOT so "
        "FOOD101_ROOT / 'images' contains class subfolders (e.g. apple_pie/, hamburger/)."
    )

transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)

train_dataset = ImageFolder(root=str(FOOD101_IMAGES), transform=transform)
CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print("num_classes:", NUM_CLASSES)

if MAX_TRAIN_SAMPLES is not None:
    train_dataset = Subset(train_dataset, range(min(MAX_TRAIN_SAMPLES, len(train_dataset))))

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

sample_images, sample_labels = next(iter(train_loader))
print("sample batch shape:", sample_images.shape)
print("sample labels:", sample_labels[:8].tolist())
print("sample classes:", [CLASS_NAMES[i] for i in sample_labels[:8].tolist()])
show_tensor_images(sample_images[:16], nrow=8, title="Food-101 training samples (resized)")


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SimpleVAE(nn.Module):
    def __init__(self, latent_channels: int = LATENT_CHANNELS):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(3, 64, stride=2),
            ConvBlock(64, 128, stride=2),
        )
        self.to_mu = nn.Conv2d(128, latent_channels, kernel_size=1)
        self.to_logvar = nn.Conv2d(128, latent_channels, kernel_size=1)

        self.decoder = nn.Sequential(
            nn.Conv2d(latent_channels, 128, kernel_size=3, padding=1),
            nn.GroupNorm(8, 128),
            nn.SiLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.GroupNorm(8, 64),
            nn.SiLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.GroupNorm(8, 32),
            nn.SiLU(),
            nn.Conv2d(32, 3, kernel_size=3, padding=1),
            nn.Tanh(),
        )

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.encoder(x)
        return self.to_mu(h), self.to_logvar(h)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


def vae_loss_fn(recon: torch.Tensor, x: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    recon_loss = F.l1_loss(recon, x)
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    loss = recon_loss + KL_WEIGHT * kl_loss
    return loss, recon_loss, kl_loss


vae = SimpleVAE().to(device)
print("latent shape:", vae.encode(sample_images[:2].to(device))[0].shape)

In [ ]:
def train_autoencoder(model: nn.Module, loader: DataLoader, epochs: int) -> None:
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_AUTOENCODER)
    model.train()

    for epoch in range(epochs):
        total_loss = 0.0
        total_recon = 0.0
        total_kl = 0.0
        total_items = 0

        for x, _ in loader:
            x = x.to(device)
            recon, mu, logvar = model(x)
            loss, recon_loss, kl_loss = vae_loss_fn(recon, x, mu, logvar)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_size = x.size(0)
            total_loss += loss.item() * batch_size
            total_recon += recon_loss.item() * batch_size
            total_kl += kl_loss.item() * batch_size
            total_items += batch_size

        print(
            f"[VAE] epoch {epoch + 1:02d}/{epochs} "
            f"loss={total_loss / total_items:.4f} "
            f"recon={total_recon / total_items:.4f} "
            f"kl={total_kl / total_items:.4f}"
        )

    torch.save(model.state_dict(), AUTOENCODER_CKPT)


if AUTOENCODER_CKPT.exists():
    vae.load_state_dict(torch.load(AUTOENCODER_CKPT, map_location=device))
    print("Loaded VAE checkpoint from", AUTOENCODER_CKPT)
else:
    train_autoencoder(vae, train_loader, AUTOENCODER_EPOCHS)

vae.eval()

with torch.no_grad():
    batch = sample_images[:8].to(device)
    recon, _, _ = vae(batch)
show_tensor_images(batch, nrow=8, title="Original images")
show_tensor_images(recon, nrow=8, title="VAE reconstructions")

In [ ]:
class GaussianFourierProjection(nn.Module):
    def __init__(self, embed_dim: int, scale: float = 30.0):
        super().__init__()
        self.W = nn.Parameter(torch.randn(embed_dim // 2) * scale, requires_grad=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_proj = x[:, None] * self.W[None, :] * 2 * math.pi
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)


class Dense(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.dense = nn.Linear(input_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dense(x)[..., None, None]


class CrossAttention(nn.Module):
    def __init__(self, embed_dim: int, hidden_dim: int, context_dim: int | None = None):
        super().__init__()
        self.embed_dim = embed_dim
        self.self_attn = context_dim is None

        self.query = nn.Linear(hidden_dim, embed_dim, bias=False)
        if self.self_attn:
            self.key = nn.Linear(hidden_dim, embed_dim, bias=False)
            self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        else:
            self.key = nn.Linear(context_dim, embed_dim, bias=False)
            self.value = nn.Linear(context_dim, hidden_dim, bias=False)

    def forward(self, tokens: torch.Tensor, context: torch.Tensor | None = None) -> torch.Tensor:
        query = self.query(tokens)
        if self.self_attn:
            key = self.key(tokens)
            value = self.value(tokens)
        else:
            key = self.key(context)
            value = self.value(context)

        scores = torch.matmul(query, key.transpose(1, 2)) / math.sqrt(self.embed_dim)
        attn = torch.softmax(scores, dim=-1)
        return torch.matmul(attn, value)


class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim: int, context_dim: int):
        super().__init__()
        self.attn_self = CrossAttention(hidden_dim, hidden_dim)
        self.attn_cross = CrossAttention(hidden_dim, hidden_dim, context_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, 3 * hidden_dim),
            nn.GELU(),
            nn.Linear(3 * hidden_dim, hidden_dim),
        )

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        x = x + self.attn_self(self.norm1(x))
        x = x + self.attn_cross(self.norm2(x), context)
        x = x + self.ffn(self.norm3(x))
        return x


class SpatialTransformer(nn.Module):
    def __init__(self, hidden_dim: int, context_dim: int):
        super().__init__()
        self.transformer = TransformerBlock(hidden_dim, context_dim)

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        residual = x
        x = x.flatten(2).transpose(1, 2)
        x = self.transformer(x, context)
        x = x.transpose(1, 2).reshape(b, c, h, w)
        return x + residual

In [ ]:
def marginal_prob_std(t: torch.Tensor | float, sigma: float = SIGMA) -> torch.Tensor:
    if not torch.is_tensor(t):
        t = torch.tensor(t, device=device, dtype=torch.float32)
    else:
        t = t.to(device=device, dtype=torch.float32)
    return torch.sqrt((sigma ** (2 * t) - 1.0) / (2.0 * math.log(sigma)))


def diffusion_coeff(t: torch.Tensor | float, sigma: float = SIGMA) -> torch.Tensor:
    if not torch.is_tensor(t):
        t = torch.tensor(t, device=device, dtype=torch.float32)
    else:
        t = t.to(device=device, dtype=torch.float32)
    return sigma ** t


class LatentUNetTransformer(nn.Module):
    def __init__(
        self,
        marginal_prob_std_fn,
        in_channels: int = LATENT_CHANNELS,
        channels: list[int] | tuple[int, int, int] = (64, 128, 256),
        embed_dim: int = 256,
        context_dim: int = 256,
        n_classes: int = 101,
    ):
        super().__init__()
        self.marginal_prob_std = marginal_prob_std_fn
        self.act = nn.SiLU()

        self.time_embed = nn.Sequential(
            GaussianFourierProjection(embed_dim),
            nn.Linear(embed_dim, embed_dim),
            nn.SiLU(),
            nn.Linear(embed_dim, embed_dim),
        )
        self.class_embed = nn.Embedding(n_classes, context_dim)

        self.conv1 = nn.Conv2d(in_channels, channels[0], kernel_size=3, padding=1, bias=False)
        self.dense1 = Dense(embed_dim, channels[0])
        self.gnorm1 = nn.GroupNorm(8, channels[0])

        self.conv2 = nn.Conv2d(channels[0], channels[1], kernel_size=3, stride=2, padding=1, bias=False)
        self.dense2 = Dense(embed_dim, channels[1])
        self.gnorm2 = nn.GroupNorm(8, channels[1])
        self.attn2 = SpatialTransformer(channels[1], context_dim)

        self.conv3 = nn.Conv2d(channels[1], channels[2], kernel_size=3, stride=2, padding=1, bias=False)
        self.dense3 = Dense(embed_dim, channels[2])
        self.gnorm3 = nn.GroupNorm(8, channels[2])
        self.attn3 = SpatialTransformer(channels[2], context_dim)

        self.tconv3 = nn.ConvTranspose2d(channels[2], channels[1], kernel_size=4, stride=2, padding=1, bias=False)
        self.dense4 = Dense(embed_dim, channels[1])
        self.tgnorm3 = nn.GroupNorm(8, channels[1])

        self.tconv2 = nn.ConvTranspose2d(channels[1], channels[0], kernel_size=4, stride=2, padding=1, bias=False)
        self.dense5 = Dense(embed_dim, channels[0])
        self.tgnorm2 = nn.GroupNorm(8, channels[0])

        self.out_conv = nn.Conv2d(channels[0], in_channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        embed = self.time_embed(t)
        class_context = self.class_embed(y).unsqueeze(1)

        h1 = self.conv1(x) + self.dense1(embed)
        h1 = self.act(self.gnorm1(h1))

        h2 = self.conv2(h1) + self.dense2(embed)
        h2 = self.act(self.gnorm2(h2))
        h2 = self.attn2(h2, class_context)

        h3 = self.conv3(h2) + self.dense3(embed)
        h3 = self.act(self.gnorm3(h3))
        h3 = self.attn3(h3, class_context)

        h = self.tconv3(h3) + self.dense4(embed)
        h = self.act(self.tgnorm3(h))
        h = h + h2

        h = self.tconv2(h) + self.dense5(embed)
        h = self.act(self.tgnorm2(h))
        h = h + h1

        h = self.out_conv(h)
        return h / self.marginal_prob_std(t)[:, None, None, None]


@torch.no_grad()
def encode_latents(x: torch.Tensor) -> torch.Tensor:
    mu, _ = vae.encode(x)
    return mu


score_model = LatentUNetTransformer(marginal_prob_std, n_classes=NUM_CLASSES).to(device)


In [ ]:
def loss_fn_cond(model: nn.Module, latents: torch.Tensor, labels: torch.Tensor, marginal_prob_std_fn, eps: float = 1e-5) -> torch.Tensor:
    random_t = torch.rand(latents.shape[0], device=latents.device) * (1.0 - eps) + eps
    noise = torch.randn_like(latents)
    std = marginal_prob_std_fn(random_t)
    perturbed_latents = latents + noise * std[:, None, None, None]
    score = model(perturbed_latents, random_t, labels)
    loss = torch.mean(torch.sum((score * std[:, None, None, None] + noise) ** 2, dim=(1, 2, 3)))
    return loss


@torch.no_grad()
def euler_maruyama_sampler(
    score_model: nn.Module,
    marginal_prob_std_fn,
    diffusion_coeff_fn,
    batch_size: int,
    x_shape: tuple[int, int, int],
    num_steps: int = NUM_DIFFUSION_STEPS,
    eps: float = 1e-3,
    y: torch.Tensor | None = None,
) -> torch.Tensor:
    t = torch.ones(batch_size, device=device)
    x = torch.randn(batch_size, *x_shape, device=device) * marginal_prob_std_fn(t)[:, None, None, None]
    time_steps = torch.linspace(1.0, eps, num_steps, device=device)
    step_size = time_steps[0] - time_steps[1]

    for time_step in time_steps:
        batch_time_step = torch.ones(batch_size, device=device) * time_step
        g = diffusion_coeff_fn(batch_time_step)
        mean_x = x + (g ** 2)[:, None, None, None] * score_model(x, batch_time_step, y) * step_size
        x = mean_x + torch.sqrt(step_size) * g[:, None, None, None] * torch.randn_like(x)

    return mean_x

In [ ]:
def train_latent_diffusion(model: nn.Module, vae_model: nn.Module, loader: DataLoader, epochs: int) -> None:
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_DIFFUSION)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.98)
    vae_model.eval()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_items = 0

        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            with torch.no_grad():
                latents = encode_latents(x)

            loss = loss_fn_cond(model, latents, y, marginal_prob_std)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_size = x.size(0)
            total_loss += loss.item() * batch_size
            total_items += batch_size

        scheduler.step()
        print(
            f"[LDM] epoch {epoch + 1:02d}/{epochs} "
            f"loss={total_loss / total_items:.4f} "
            f"lr={scheduler.get_last_lr()[0]:.2e}"
        )

    torch.save(model.state_dict(), DIFFUSION_CKPT)


if DIFFUSION_CKPT.exists():
    score_model.load_state_dict(torch.load(DIFFUSION_CKPT, map_location=device))
    print("Loaded diffusion checkpoint from", DIFFUSION_CKPT)
else:
    train_latent_diffusion(score_model, vae, train_loader, DIFFUSION_EPOCHS)

score_model.eval()

In [ ]:
@torch.no_grad()
def sample_class(model: nn.Module, vae_model: nn.Module, class_id: int, n_samples: int = 8) -> torch.Tensor:
    labels = torch.full((n_samples,), class_id, device=device, dtype=torch.long)
    latent_samples = euler_maruyama_sampler(
        score_model=model,
        marginal_prob_std_fn=marginal_prob_std,
        diffusion_coeff_fn=diffusion_coeff,
        batch_size=n_samples,
        x_shape=(LATENT_CHANNELS, LATENT_SIZE, LATENT_SIZE),
        y=labels,
    )
    decoded = vae_model.decode(latent_samples)
    return decoded


# Pick classes that exist in Food-101 (folder names)
target_classes = ["apple_pie", "hamburger", "pizza", "sushi"]
all_samples = []

for class_name in target_classes:
    if class_name not in CLASS_NAMES:
        raise ValueError(f"class {class_name!r} not in dataset; pick from CLASS_NAMES")
    class_id = CLASS_NAMES.index(class_name)
    samples = sample_class(score_model, vae, class_id=class_id, n_samples=4)
    all_samples.append(samples)
    show_tensor_images(samples, nrow=4, title=f"Conditional samples for class: {class_name}")


## Notes

- **Food-101** replaces CIFAR-10: same latent-diffusion pipeline, larger **64×64** inputs and **101** classes.
- Conditioning is still **class ID / class name** (`ImageFolder` label order matches `CLASS_NAMES`).
- Checkpoints live under `checkpoints/stable_diffusion_hw_food101/` so they do not overwrite the CIFAR-10 run.
- If VRAM is tight, lower `BATCH_SIZE` or `IMAGE_SIZE` (if you change `IMAGE_SIZE`, keep `LATENT_SIZE = IMAGE_SIZE // 4` and the same VAE stride pattern: two ×2 downsamples).
- For faster iteration, set `MAX_TRAIN_SAMPLES` in the first code cell.
